## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")


In [4]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

d:\2026-courses\agenticai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B6B2CBFF70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B6B2CEC4F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(
    content='Hi, My name is Bilal and I am a chief AI Engineer at a company called Agentic AI')
])

AIMessage(content="Hello Bilal, it's nice to meet you. As the Chief AI Engineer at Agentic AI, I'm sure you're working on some exciting projects. Can you tell me a bit more about what Agentic AI does and what kind of AI-related work you're involved in?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 56, 'total_tokens': 114, 'completion_time': 0.084327272, 'completion_tokens_details': None, 'prompt_time': 0.004574415, 'prompt_tokens_details': None, 'queue_time': 0.05187945, 'total_time': 0.088901687}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0b9a-585f-77a1-80d8-cc572a7dbdc3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 58, 'total_tokens': 114})

In [6]:
from langchain_core.messages import HumanMessage, AIMessage

model.invoke(
             [
                HumanMessage( content='Hi, My name is Bilal and I am a chief AI Engineer at a company called Agentic AI'),
                AIMessage(content="Nice to meet you, Bilal. Congratulations on being the Chief AI Engineer at Agentic AI. That sounds like a fascinating role. I'd be happy to learn more about your work and the company. What areas of AI are you and your team focusing on, and what are some of the exciting projects you're currently working on?"),
                HumanMessage(content="Hey What is my name and what do I do?")
             ]
)








AIMessage(content='Your name is Bilal, and you are the Chief AI Engineer at a company called Agentic AI.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 144, 'total_tokens': 166, 'completion_time': 0.02925673, 'completion_tokens_details': None, 'prompt_time': 0.010523404, 'prompt_tokens_details': None, 'queue_time': 0.045077814, 'total_time': 0.039780134}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0b9a-59d3-71c3-8c20-bdc570ad57e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_tokens': 22, 'total_tokens': 166})

## Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:id)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory(session_id=session_id)
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model, get_session_history)


d:\2026-courses\agenticai\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
config = {
    "configurable":{
        "session_id":"chat1"
    }
}

In [9]:
response = with_message_history.invoke(
    [HumanMessage(content='Hi, My name is Bilal and I am a chief AI Engineer at a company called Agentic AI'),],
    config=config
)

In [10]:
response.content

"Nice to meet you, Bilal. It's impressive that you hold the position of Chief AI Engineer at Agentic AI. Can you tell me more about the work that Agentic AI does and what some of the exciting projects you've been working on lately?"

In [11]:
with_message_history.invoke(
    [HumanMessage(content="Hey What is my name and what do I do?")],
    config=config
)

AIMessage(content="Your name is Bilal, and you're the Chief AI Engineer at Agentic AI.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 129, 'total_tokens': 148, 'completion_time': 0.021235402, 'completion_tokens_details': None, 'prompt_time': 0.017115533, 'prompt_tokens_details': None, 'queue_time': 0.04473882, 'total_time': 0.038350935}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e09ee421cf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0b9a-5e45-7230-8ba5-34f86be106eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 129, 'output_tokens': 19, 'total_tokens': 148})

In [12]:
## Change the session id
config1 = {
    "configurable":{
        "session_id":"chat2"
    }
}

response= with_message_history.invoke(
    [HumanMessage(content="Hey What is my name and what do I do?")],
    config=config1
)

In [13]:
response.content

"I'm happy to chat with you. However, I'm a large language model, I don't have any prior knowledge about you, so I won't be able to recall your name or occupation. \n\nThis conversation just started, and I don't have any information about you. But feel free to share your name and what you do, and I'd be happy to chat with you."

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [15]:
chain.invoke(
    {
        "messages":[HumanMessage(content="Hey my name is Bilal")]
    }
)

AIMessage(content="Nice to meet you, Bilal. I'm here to help you with any questions or topics you'd like to discuss. What's on your mind today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 58, 'total_tokens': 91, 'completion_time': 0.046541543, 'completion_tokens_details': None, 'prompt_time': 0.004878587, 'prompt_tokens_details': None, 'queue_time': 0.048576871, 'total_time': 0.05142013}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e09ee421cf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0b9a-6131-7680-8b14-92402b9e9722-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 33, 'total_tokens': 91})

In [16]:
with_message_history=RunnableWithMessageHistory(chain, get_session_history)

d:\2026-courses\agenticai\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
config2 = {
    "configurable":{
        "session_id":"chat3"
    }
}

In [18]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey my name is Bilal")],
    config=config2
)
response.content

"Nice to meet you, Bilal. I'm happy to assist you with any questions or information you may need. Is there something specific on your mind, or would you like to start a conversation?"

In [19]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Bilal.'

In [20]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [21]:
response = chain.invoke(
    {
        "language":"Spanish",
        "messages":[HumanMessage(content="My name is Bilal")]
    }
)
response.content




    

'Encantado de conocerte, Bilal. ¿En qué puedo ayudarte hoy?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [22]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [23]:
config3 = {
    "configurable":{
        "session_id":"chat4"
    }
}

In [24]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Bilal")],"language":"Spanish"},
    config=config3
)
repsonse.content





'Hola Bilal, soy su asistente. ¿En qué puedo ayudarte hoy?'

In [25]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Spanish"},
    config=config,
)

In [26]:
response.content

'Tu nombre es Bilal. ¿Te gusta ese nombre?'

### Managing the Conversation History
- One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model.

- The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [27]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45, #max number of tokens to keep
    strategy="last", #keep the last n messages
    token_counter=model, #use the model to count tokens
    include_system=True, #always keep the system message
    allow_partial=False, #don't allow partial messages
    start_on="human" #start trimming from the human message
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm Bilal"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)


d:\2026-courses\agenticai\venv\lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [28]:
## Pass the trimmer in chain

from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"I'm not sure, I don't have any information about your preferences. Can you tell me what your favorite ice cream flavor is?"

In [29]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked for the solution to the math problem "2 + 2".'

In [30]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config4={"configurable":{"session_id":"chat5"}}

d:\2026-courses\agenticai\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [31]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't mention your name. I'm happy to chat with you though!"

In [32]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't ask a math problem. Our conversation started with a question about having fun, and then we discussed your name. If you'd like to ask a math problem, I'd be happy to help!"

## Working with vectors and retriever
### Document
Langchain implements a Document abstraction, which is intended to represent a unit text and associated medata. It has two attributes:
 - page_content= a string representing the content.
 - metadata: a dict containing arbitrary metadata. The metadata attribute can capture information about the source of information of the document, its relationship to other documents, and other information. Note that an individual Document object often reprsents a chunk of large document.

In [33]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness",
        metadat ={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Cats are known independent pets that often enjoy their own space",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiering relatively simple care",
        metadata={"source":"fish-pets-doc"}
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech",
        metadata={"source":"bird-pets-doc"}
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around",
        metadata={"source":"mammal-pets-doc"}
    )
]




In [34]:
documents

[Document(metadata={}, page_content='Dogs are great companions, known for their loyalty and friendliness'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are known independent pets that often enjoy their own space'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiering relatively simple care'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around')]

In [36]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B6B4132050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B6B4133190>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [37]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2287.49it/s]


In [38]:
## Vector Store
from langchain_chroma import Chroma

vector_store=Chroma.from_documents(
    documents,
    embedding=embeddings,
)

In [40]:
vector_store.similarity_search("What are the best pets for beginners?")

[Document(id='10a46384-3edf-4ae9-9921-ad6f74cfb5a3', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiering relatively simple care'),
 Document(id='8405522c-42ed-4c57-af51-151b8109c785', metadata={}, page_content='Dogs are great companions, known for their loyalty and friendliness'),
 Document(id='233d61a8-e6b0-4c6d-8341-2936e2eee62e', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are known independent pets that often enjoy their own space'),
 Document(id='ebe1c8d4-69eb-4f50-a8fe-4ec648c298a9', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech')]

In [41]:
## Async Query
await vector_store.asimilarity_search("What are the best pets for beginners?")

[Document(id='10a46384-3edf-4ae9-9921-ad6f74cfb5a3', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiering relatively simple care'),
 Document(id='8405522c-42ed-4c57-af51-151b8109c785', metadata={}, page_content='Dogs are great companions, known for their loyalty and friendliness'),
 Document(id='233d61a8-e6b0-4c6d-8341-2936e2eee62e', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are known independent pets that often enjoy their own space'),
 Document(id='ebe1c8d4-69eb-4f50-a8fe-4ec648c298a9', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech')]

In [42]:
vector_store.similarity_search_with_score("What are the best pets for beginners?")

[(Document(id='10a46384-3edf-4ae9-9921-ad6f74cfb5a3', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiering relatively simple care'),
  0.7375314235687256),
 (Document(id='8405522c-42ed-4c57-af51-151b8109c785', metadata={}, page_content='Dogs are great companions, known for their loyalty and friendliness'),
  1.2429436445236206),
 (Document(id='233d61a8-e6b0-4c6d-8341-2936e2eee62e', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are known independent pets that often enjoy their own space'),
  1.2661006450653076),
 (Document(id='ebe1c8d4-69eb-4f50-a8fe-4ec648c298a9', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech'),
  1.5625951290130615)]

### Retrivers
- LangChain VectrorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language Chains.
- Langchain Retrievers are Runnables, so they implement a standard set of methods (synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains
- We create a simple vesrion of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below, we'll build around the similarity_search method:

In [ ]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(["cats", "dog"])


[[Document(id='233d61a8-e6b0-4c6d-8341-2936e2eee62e', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are known independent pets that often enjoy their own space')],
 [Document(id='8405522c-42ed-4c57-af51-151b8109c785', metadata={}, page_content='Dogs are great companions, known for their loyalty and friendliness')]]

In [53]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cats", "dog"])

[[Document(id='233d61a8-e6b0-4c6d-8341-2936e2eee62e', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are known independent pets that often enjoy their own space')],
 [Document(id='8405522c-42ed-4c57-af51-151b8109c785', metadata={}, page_content='Dogs are great companions, known for their loyalty and friendliness')]]

In [ ]:
# Integrate the retriver with chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message= """
Answer this question using the provided context only.
{question}
context:
{context}
"""
prompt = ChatPromptTemplate.from_messages(
    [
        "human", message
    ]
)
rag_chain={"context":retriever, "question":RunnablePassthrough()}|prompt|model
response=rag_chain.invoke("Tell me about Dogs")
response.content

'Dogs are great companions, known for their loyalty and friendliness.'